In [1]:
import math
import random

DAYS_PER_YEAR = 252
DAYS_PER_WEEK = 5
STEPS_PER_DAY = 4

def simulate_prices(days: int):
    prices_len = STEPS_PER_DAY * days
    prices = [50]
    dt = 1 / (DAYS_PER_YEAR * STEPS_PER_DAY)
    sigma = 2.51  # 251% volatility
    for _ in range(prices_len):
        z = random.gauss(0, 1)
        # GBM step with zero drift
        prev = prices[-1]
        next_price = prev * math.exp(-0.5 * sigma**2 * dt + sigma * math.sqrt(dt) * z)
        prices.append(next_price)
    return prices

def simulate_final_prices(days: int, n_sims: int):
    finals = []
    for _ in range(n_sims):
        path = simulate_prices(days)
        finals.append(path[-1])
    return finals

def call_payoff(final_price: float, strike: float):
    return max(final_price - strike, 0.0)

def put_payoff(final_price: float, strike: float):
    return max(strike - final_price, 0.0)

def evaluate_call_price(option_price: float, finals, strike: float):
    # Estimates E[payoff - option_price]
    total = 0.0
    for S_T in finals:
        payoff = call_payoff(S_T, strike)
        total += (payoff - option_price)
    return total / len(finals)

def evaluate_put_price(option_price: float, finals, strike: float):
    # Estimates E[payoff - option_price]
    total = 0.0
    for S_T in finals:
        payoff = put_payoff(S_T, strike)
        total += (payoff - option_price)
    return total / len(finals)

def find_call_option_price(finals, strike: float, iterations: int = 20):
    low, high = 0.0, 1e3  # initial bounds
    for _ in range(iterations):
        mid = 0.5 * (low + high)
        value = evaluate_call_price(mid, finals, strike)
        if value > 0:
            low = mid
        else:
            high = mid
    return 0.5 * (low + high)

def find_put_option_price(finals, strike: float, iterations: int = 20):
    low, high = 0.0, 1e3 # initial bounds
    for _ in range(iterations):
        mid = 0.5 * (low + high)
        value = evaluate_put_price(mid, finals, strike)
        if value > 0:
            low = mid
        else:
            high = mid
    return 0.5 * (low + high)

def probability_call_itm(finals, strike: float):
    count = sum(1 for S_T in finals if S_T > strike)
    return count / len(finals)

def probability_put_itm(finals, strike: float):
    count = sum(1 for S_T in finals if S_T < strike)
    return count / len(finals)

def analyze_option(days: int, strike: int, n_sims: int, is_call: bool, print_output: bool = False):
    finals = simulate_final_prices(days, n_sims)
    if is_call:
        prob = probability_call_itm(finals, strike)
        price = find_call_option_price(finals, strike)
    else:
        prob = probability_put_itm(finals, strike)
        price = find_put_option_price(finals, strike)
    is_call_str = "Call" if is_call else "Put"
    
    if print_output:
        print(f"---")
        print(f"Option: Strike {strike} {is_call_str}, {days} Days")
        print(f"Probability ITM: {prob:.4f}")
        print(f"Estimated option price: {price:.4f}")
    return prob, price

In [2]:
# get ts going
tte_list = [3 * DAYS_PER_WEEK for _ in range(8)]
tte_list[6] -= DAYS_PER_WEEK
tte_list[7] -= DAYS_PER_WEEK
strike_list = [50, 50, 35, 40, 45, 60, 50, 50]
is_call_list = [False, True, False, False, False, True, False, True]

In [3]:
# normal option analysis, no exotic shit yet

# SIMS = 500000
# for i in range(8):
#     analyze_option(tte_list[i], strike_list[i], SIMS, is_call_list[i])

In [4]:
# only the 2 week options seem mispriced... but there's no way to get a distribution!

import numpy as np
from statistics import NormalDist

def get_distr(days: int, strike: int, m_sims: int, n_sims: int, is_call: bool):
    probs = np.array([])
    prices = np.array([])
    for _ in range(m_sims):
        prob, price = analyze_option(days, strike, n_sims, is_call)
        probs = np.append(probs, prob)
        prices = np.append(prices, price)
    return probs.mean(), probs.std(), prices.mean(), prices.std()

def price_conf_int(days: int, strike: int, m_sims: int, n_sims: int, is_call: bool, conf: float, print_output: bool = False):
    _, _, s_mean, s_std = get_distr(days, strike, m_sims, n_sims, is_call)
    z_score = NormalDist().inv_cdf(conf + (1 - conf) / 2)
    is_call_str = "Call" if is_call else "Put"
    low = s_mean - z_score * s_std
    high = s_mean + z_score * s_std
    
    if print_output:
        print(f"---")
        print(f"Option: Strike {strike} {is_call_str}, {days} Days")
        print(f"Low:  {low}")
        print(f"High: {high}")
    return low, high

In [5]:
# there's market type shit
bids = [12, 12, 4.33, 6.5, 9.05, 8.8, 9.7, 9.7]
asks = [12.05, 12.05, 4.35, 6.55, 9.1, 8.85, 9.75, 9.75]

In [9]:
M_SIMS = 200
N_SIMS = 1000

for i in range(8):
    _, _, s_mean, s_std = get_distr(tte_list[i], strike_list[i], M_SIMS, N_SIMS, is_call_list[i])
    target_bid, target_ask = bids[i], asks[i]
    low_z = (target_bid - s_mean) / s_std
    high_z = (target_ask - s_mean) / s_std
    low_prob = NormalDist().cdf(low_z)
    high_prob = 1 - NormalDist().cdf(high_z)

    print(f"---")
    print(f"Option {i}")
    print(f"{round(s_mean, 4)}, {round(s_std, 4)}")
    print(f"{round(low_prob, 4)}, {round(high_prob, 4)}")

---
Option 0
12.0323, 0.415
0.469, 0.483
---
Option 1
11.985, 0.8547
0.507, 0.4697
---
Option 2
4.3258, 0.2311
0.5073, 0.4582
---
Option 3
6.5202, 0.2824
0.4715, 0.458
---
Option 4
9.0952, 0.3066
0.4413, 0.4938
---
Option 5
8.7208, 0.771
0.5409, 0.4335
---
Option 6
9.8876, 0.3369
0.2888, 0.6585
---
Option 7
9.8712, 0.6175
0.3908, 0.5778


In [13]:
# to figure out Kelly betting, we gotta get avg z-score above / below a threshold

def expected_below_normal(z_th: float, sims: int):
    z_lower_area = NormalDist().cdf(z_th)
    trials = np.array([])
    for _ in range(sims):
        rand_area = np.random.uniform(0, z_lower_area)
        rand_z = NormalDist().inv_cdf(rand_area)
        trials = np.append(trials, rand_z)
    return trials.mean()

def expected_above_normal(z_th: float, sims: int):
    z_upper_area = 1 - NormalDist().cdf(z_th)
    trials = np.array([])
    for _ in range(sims):
        rand_area = np.random.uniform(0, z_upper_area)
        rand_z = -NormalDist().inv_cdf(rand_area)
        trials = np.append(trials, rand_z)
    return trials.mean()

def kelly(p: float, a: float, b: float):
    return p/a - (1-p)/b

In [ ]:
M_SIMS = 200
N_SIMS = 1000
Z_SIMS = 10000

for i in [6, 7]:
    _, _, s_mean, s_std = get_distr(tte_list[i], strike_list[i], M_SIMS, N_SIMS, is_call_list[i])
    target_bid, target_ask = bids[i], asks[i]
    low_z = (target_bid - s_mean) / s_std
    high_z = (target_ask - s_mean) / s_std
    
    above_ev = expected_above_normal(high_z, Z_SIMS) * s_std + s_mean
    below_ev = expected_below_normal(high_z, Z_SIMS) * s_std + s_mean
    ask = asks[i]
    
    p = 1 - NormalDist().cdf(high_z)
    a = (ask - below_ev) / ask
    b = (above_ev - ask) / ask
    k = kelly(p, a, b)

    print(f"---")
    print(f"Option {i}")
    print(f"Parameters: {round(p, 4)}, {round(a, 4)}, {round(b, 4)}")
    print(f"Kelly bet: {round(k, 4)}")

# kelly bets are ~20 and ~4-5, which seems odd...

---
Option 6
Parameters: 0.6568, 0.0219, 0.0315
Kelly bet: 19.0662
---
Option 7
Parameters: 0.5793, 0.0468, 0.0571
Kelly bet: 5.0141


In [40]:
# ok chatgpt type shi

from copy import deepcopy

def compute_kelly(days: int, strike: int, m_sims: int, n_sims: int, is_call: bool, price: float, print_output: bool = False):
    finals = simulate_final_prices(days, n_sims)
    returns = [(max(S_T - strike, 0) - price) / price for S_T in finals]

    def objective(f: float):
        copy_returns = deepcopy(returns)
        return np.mean(np.log(1 + f * np.array(copy_returns)))
    
    answers = np.array([])
    for _ in range(m_sims):
        fs = np.linspace(0, 1, 1000)  # cap at 100% bankroll for sanity
        vals = [objective(f) for f in fs]
        final_answer = fs[np.argmax(vals)]
        answers = np.append(answers, final_answer)
    
    kelly = answers.mean()
    is_call_str = "Call" if is_call else "Put"
    if print_output:
        print(f"---")
        print(f"Option: Strike {strike} {is_call_str}, {days} Days")
        print(f"Kelly Bet: {round(kelly, 6)}")
    
    return kelly

In [ ]:
M_SIMS = 100
N_SIMS = 1000

for i in [6, 7]:
    compute_kelly(tte_list[i], strike_list[i], M_SIMS, N_SIMS, is_call_list[i], asks[i], True)

In [55]:
def test_straddle_50(days: int, m_sims: int, n_sims: int, print_output: bool = False):
    strike = 50

    pnl_means = np.array([])
    pnl_stds = np.array([])
    prob_profit = np.array([])

    for _ in range(m_sims):
        # simulate paths
        finals = simulate_final_prices(days, n_sims)

        # price each leg using SAME finals (important for consistency)
        call_price = asks[1] # find_call_option_price(finals, strike)
        put_price = asks[0] # find_put_option_price(finals, strike)

        total_cost = call_price + put_price

        # compute PnL distribution for this trial
        pnls = []
        for S_T in finals:
            payoff = call_payoff(S_T, strike) + put_payoff(S_T, strike)
            pnl = payoff - total_cost
            pnls.append(pnl)

        pnls = np.array(pnls)

        pnl_means = np.append(pnl_means, pnls.mean())
        pnl_stds = np.append(pnl_stds, pnls.std())
        prob_profit = np.append(prob_profit, np.mean(pnls > 0))

    # aggregate across trials
    mean_pnl = pnl_means.mean()
    std_pnl = pnl_means.std()
    avg_std = pnl_stds.mean()
    avg_prob_profit = prob_profit.mean()

    if print_output:
        print(f"---")
        print(f"50-50 Straddle, {days} Days")
        print(f"Mean PnL: {round(mean_pnl, 4)}")
        print(f"PnL Std (across trials): {round(std_pnl, 4)}")
        print(f"Avg Path Std: {round(avg_std, 4)}")
        print(f"Probability Profit: {round(avg_prob_profit, 4)}")

    return mean_pnl, std_pnl, avg_std, avg_prob_profit

In [56]:
M_SIMS = 200
N_SIMS = 1000

test_straddle_50(15, M_SIMS, N_SIMS, print_output=True)

---
50-50 Straddle, 15 Days
Mean PnL: -0.008
PnL Std (across trials): 0.7215
Avg Path Std: 23.6245
Probability Profit: 0.3921


(np.float64(-0.008044991417257967),
 np.float64(0.7215424549207285),
 np.float64(23.624535079041497),
 np.float64(0.3921299999999999))

In [57]:
def test_strangle_45_60(days: int, m_sims: int, n_sims: int, print_output: bool = False):
    put_strike = 45
    call_strike = 60

    pnl_means = np.array([])
    pnl_stds = np.array([])
    prob_profit = np.array([])

    for _ in range(m_sims):
        # simulate paths
        finals = simulate_final_prices(days, n_sims)

        # price each leg using SAME finals
        call_price = asks[5] # find_call_option_price(finals, call_strike)
        put_price = asks[4] # find_put_option_price(finals, put_strike)

        total_cost = call_price + put_price

        # compute PnL distribution
        pnls = []
        for S_T in finals:
            payoff = call_payoff(S_T, call_strike) + put_payoff(S_T, put_strike)
            pnl = payoff - total_cost
            pnls.append(pnl)

        pnls = np.array(pnls)

        pnl_means = np.append(pnl_means, pnls.mean())
        pnl_stds = np.append(pnl_stds, pnls.std())
        prob_profit = np.append(prob_profit, np.mean(pnls > 0))

    # aggregate across trials
    mean_pnl = pnl_means.mean()
    std_pnl = pnl_means.std()
    avg_std = pnl_stds.mean()
    avg_prob_profit = prob_profit.mean()

    if print_output:
        print(f"---")
        print(f"45-60 Strangle, {days} Days")
        print(f"Mean PnL: {round(mean_pnl, 4)}")
        print(f"PnL Std (across trials): {round(std_pnl, 4)}")
        print(f"Avg Path Std: {round(avg_std, 4)}")
        print(f"Probability Profit: {round(avg_prob_profit, 4)}")

    return mean_pnl, std_pnl, avg_std, avg_prob_profit

In [58]:
M_SIMS = 200
N_SIMS = 1000

test_strangle_45_60(15, M_SIMS, N_SIMS, print_output=True)

---
45-60 Strangle, 15 Days
Mean PnL: 0.0001
PnL Std (across trials): 0.6724
Avg Path Std: 22.5717
Probability Profit: 0.3951


(np.float64(0.00011958105171572364),
 np.float64(0.6723937961970052),
 np.float64(22.571686190711553),
 np.float64(0.39506))

In [ ]:
from math import floor, ceil, log, sqrt, exp
from statistics import NormalDist

class BlackScholes:
    @staticmethod
    def black_scholes_call(spot, strike, time_to_expiry, volatility):
        d1 = (log(spot) - log(strike) + (0.5 * volatility * volatility) * time_to_expiry) / (volatility * sqrt(time_to_expiry))
        d2 = d1 - volatility * sqrt(time_to_expiry)
        call_price = spot * NormalDist().cdf(d1) - strike * NormalDist().cdf(d2)
        return call_price

    @staticmethod
    def black_scholes_put(spot, strike, time_to_expiry, volatility):
        d1 = (log(spot) - log(strike) + (0.5 * volatility * volatility) * time_to_expiry) / (volatility * sqrt(time_to_expiry))
        d2 = d1 - volatility * sqrt(time_to_expiry)
        put_price = strike * NormalDist().cdf(-d2) - spot * NormalDist().cdf(-d1)
        return put_price

    @staticmethod
    def delta_call(spot, strike, time_to_expiry, volatility):
        d1 = (log(spot) - log(strike) + (0.5 * volatility * volatility) * time_to_expiry) / (volatility * sqrt(time_to_expiry))
        return NormalDist().cdf(d1)
    
    @staticmethod
    def delta_put(spot, strike, time_to_expiry, volatility):
        return BlackScholes.delta_call(spot, strike, time_to_expiry, volatility) - 1

    @staticmethod
    def gamma(spot, strike, time_to_expiry, volatility):
        d1 = (log(spot) - log(strike) + (0.5 * volatility * volatility) * time_to_expiry) / (volatility * sqrt(time_to_expiry))
        return NormalDist().pdf(d1) / (spot * volatility * sqrt(time_to_expiry))

    @staticmethod
    def vega(spot, strike, time_to_expiry, volatility):
        d1 = (log(spot) - log(strike) + (0.5 * volatility * volatility) * time_to_expiry) / (volatility * sqrt(time_to_expiry))
        return NormalDist().pdf(d1) * (spot * sqrt(time_to_expiry)) / 100

    @staticmethod
    def implied_volatility_call(
        call_price, spot, strike, time_to_expiry, max_iterations=200, tolerance=1e-10
    ):
        low_vol = 0.0001
        high_vol = 2.0
        volatility = (low_vol + high_vol) / 2.0
        for _ in range(max_iterations):
            estimated_price = BlackScholes.black_scholes_call(spot, strike, time_to_expiry, volatility)
            diff = estimated_price - call_price
            if abs(diff) < tolerance:
                break
            elif diff > 0:
                high_vol = volatility
            else:
                low_vol = volatility
            volatility = (low_vol + high_vol) / 2.0
        return volatility
    
    @staticmethod
    def implied_volatility_put(
        put_price, spot, strike, time_to_expiry, max_iterations=200, tolerance=1e-10
    ):
        low_vol = 0.0001
        high_vol = 2.0
        volatility = (low_vol + high_vol) / 2.0
        for _ in range(max_iterations):
            estimated_price = BlackScholes.black_scholes_put(spot, strike, time_to_expiry, volatility)
            diff = estimated_price - put_price
            if abs(diff) < tolerance:
                break
            elif diff > 0:
                high_vol = volatility
            else:
                low_vol = volatility
            volatility = (low_vol + high_vol) / 2.0
        return volatility

    @staticmethod
    def moneyness(spot, strike, time_to_expiry):
        return log(spot / strike) / sqrt(time_to_expiry)

In [70]:
del1 = BlackScholes.delta_call(50, 60, 3 * DAYS_PER_WEEK / DAYS_PER_YEAR, 2.51)
del2 = BlackScholes.delta_put(50, 45, 3 * DAYS_PER_WEEK / DAYS_PER_YEAR, 2.51)
del3 = BlackScholes.delta_call(50, 50, 2 * DAYS_PER_WEEK / DAYS_PER_YEAR, 2.51)
del4 = BlackScholes.delta_put(50, 50, 2 * DAYS_PER_WEEK / DAYS_PER_YEAR, 2.51)
del_avg = (del1 + del2 + del3 + del4)/4

print(del1)
print(del2)
print(del3)
print(del4)
print(del_avg)

0.503375472311103
-0.3162395967716509
0.5987070928783335
-0.40129290712166654
0.09613751532402975


In [71]:
def binary_put_payoff(price: float, strike: float, fv: float = 50):
    if price <= strike:
        return fv - strike
    else:
        return 0

def evaluate_binary_put_price(option_price: float, finals, strike: float):
    # Estimates E[payoff - option_price]
    total = 0.0
    for S_T in finals:
        payoff = binary_put_payoff(S_T, strike)
        total += (payoff - option_price)
    return total / len(finals)

def find_binary_put_option_price(finals, strike: float, iterations: int = 20):
    low, high = 0.0, 1e3 # initial bounds
    for _ in range(iterations):
        mid = 0.5 * (low + high)
        value = evaluate_binary_put_price(mid, finals, strike)
        if value > 0:
            low = mid
        else:
            high = mid
    return 0.5 * (low + high)

def analyze_binary_put(days: int, strike: int, n_sims: int, print_output: bool = False):
    finals = simulate_final_prices(days, n_sims)
    prob = probability_put_itm(finals, strike)
    price = find_binary_put_option_price(finals, strike)
    
    if print_output:
        print(f"---")
        print(f"Option: Strike {strike} Binary Put, {days} Days")
        print(f"Probability ITM: {prob:.4f}")
        print(f"Estimated option price: {price:.4f}")
    return prob, price

In [73]:
analyze_binary_put(3 * DAYS_PER_WEEK, 40, 100000, True)

---
Option: Strike 40 Binary Put, 15 Days
Probability ITM: 0.4790
Estimated option price: 4.7898


(0.47897, 4.789829254150391)

In [74]:
def chooser_payoff(path, strike: float, decision_step: int):
    S_decision = path[decision_step]
    S_T = path[-1]

    if S_decision >= strike:
        return max(S_T - strike, 0.0)  # call
    else:
        return max(strike - S_T, 0.0)  # put


def evaluate_chooser_price(option_price: float, paths, strike: float, decision_step: int):
    total = 0.0
    for path in paths:
        payoff = chooser_payoff(path, strike, decision_step)
        total += (payoff - option_price)
    return total / len(paths)


def find_chooser_option_price(paths, strike: float, decision_step: int, iterations: int = 20):
    low, high = 0.0, 1e3

    for _ in range(iterations):
        mid = 0.5 * (low + high)
        value = evaluate_chooser_price(mid, paths, strike, decision_step)

        if value > 0:
            low = mid
        else:
            high = mid

    return 0.5 * (low + high)


def probability_chooser_profit(paths, strike: float, decision_step: int, price: float):
    count = 0
    for path in paths:
        payoff = chooser_payoff(path, strike, decision_step)
        if payoff > price:
            count += 1
    return count / len(paths)


def analyze_chooser_option(days: int, strike: int, n_sims: int, print_output: bool = False):
    # simulate full paths
    paths = []
    for _ in range(n_sims):
        paths.append(simulate_prices(days))

    # decision at 2 weeks
    decision_days = 2 * DAYS_PER_WEEK
    decision_step = decision_days * STEPS_PER_DAY

    # price the chooser
    price = find_chooser_option_price(paths, strike, decision_step)

    # probability of profit (optional but consistent with your framework)
    prob_profit = probability_chooser_profit(paths, strike, decision_step, price)

    if print_output:
        print(f"---")
        print(f"Chooser Option: Strike {strike}, {days} Days")
        print(f"Decision at day {decision_days}")
        print(f"Estimated price: {price:.4f}")
        print(f"Probability Profit: {prob_profit:.4f}")

    return prob_profit, price

In [79]:
analyze_chooser_option(3 * DAYS_PER_WEEK, 50, 100000, True)

---
Chooser Option: Strike 50, 15 Days
Decision at day 10
Estimated price: 21.9388
Probability Profit: 0.4227


(0.42272, 21.938800811767578)